In [10]:
# ---------------------------------------------------------------------
# CELL 1: SETUP
# Unified master input dataset builder
# scripts/master_inputs/create.ipynb
# ---------------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
import json
import warnings

import numpy as np
import pandas as pd
import rasterio

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------------------------------------------------------------
# ROOT PATHS
# ---------------------------------------------------------------------
PROJECT_ROOT = Path(r"C:\Projects\Infer RozviDrought").resolve()

DATA_DIR = PROJECT_ROOT / "data"
RASTERS_DIR = PROJECT_ROOT / "rasters"

ERA5_DIR = DATA_DIR / "simulated" / "era5_land"
CMIP6_CORRECTED_DIR = RASTERS_DIR / "corrected" / "cmip6"
PROXY_CORRECTED_DIR = RASTERS_DIR / "corrected" / "proxies"

ARTIFACTS_DIR = DATA_DIR / "artifacts"
DATASETS_DIR = ARTIFACTS_DIR / "datasets"
MANIFESTS_DIR = ARTIFACTS_DIR / "manifests"

DATASETS_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------
# CONFIG
# ---------------------------------------------------------------------
RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID = f"master_inputs_dataset_{RUN_TS}"

SCENARIOS = ["ssp245", "ssp370", "ssp585"]

HIST_START = "198109"
HIST_END = "202512"
FUTURE_START = "202601"
FUTURE_END = "205012"

MASTER_DATASET_PATH = DATASETS_DIR / f"{RUN_ID}.parquet"
MASTER_MANIFEST_PATH = MANIFESTS_DIR / f"{RUN_ID}_manifest.json"

# ---------------------------------------------------------------------
# SOURCE PATHS
# ---------------------------------------------------------------------
BASE_VARIABLES = {
    "t2m": ERA5_DIR / "t2m",
    "d2m": ERA5_DIR / "d2m",
    "sm": ERA5_DIR / "sm",
    "pet": ERA5_DIR / "pet",
}

PROXY_HISTORICAL = {
    "ndvi": PROXY_CORRECTED_DIR / "ndvi" / "historical",
    "tws": PROXY_CORRECTED_DIR / "tws" / "historical",
}

FUTURE_CMIP6 = {
    "tas": "tas",
    "d2m": "d2m",
    "sm": "sm",
    "pet": "pet",
}

FUTURE_PROXIES = {
    "ndvi": "ndvi",
    "tws": "tws",
}

# ---------------------------------------------------------------------
# VARIABLE METADATA
# ---------------------------------------------------------------------
VARIABLE_SPECS = {
    "t2m": {
        "standard_name": "air_temperature",
        "long_name": "2 metre air temperature",
        "units": "K",
        "historical_source_type": "observed_reference",
        "future_source_type": "bias_corrected_cmip6",
        "historical_folder_key": "t2m",
        "future_folder_key": "tas",
        "method_note": "Historical from ERA5-Land reference inputs; future from CMIP6 tas bias-corrected to the inference grid.",
    },
    "d2m": {
        "standard_name": "dew_point_temperature",
        "long_name": "2 metre dew point temperature",
        "units": "K",
        "historical_source_type": "observed_reference",
        "future_source_type": "bias_corrected_cmip6",
        "historical_folder_key": "d2m",
        "future_folder_key": "d2m",
        "method_note": "Historical from ERA5-Land reference inputs; future derived from CMIP6 humidity and bias-corrected.",
    },
    "sm": {
        "standard_name": "soil_moisture",
        "long_name": "soil moisture",
        "units": "m3 m-3",
        "historical_source_type": "observed_reference",
        "future_source_type": "bias_corrected_cmip6",
        "historical_folder_key": "sm",
        "future_folder_key": "sm",
        "method_note": "Historical from reference inputs; future from CMIP6 soil moisture bias-corrected to the inference grid.",
    },
    "pet": {
        "standard_name": "potential_evapotranspiration",
        "long_name": "potential evapotranspiration",
        "units": "mm month-1",
        "historical_source_type": "observed_reference",
        "future_source_type": "bias_corrected_cmip6",
        "historical_folder_key": "pet",
        "future_folder_key": "pet",
        "method_note": "Historical from ERA5 reference inputs with positive sign convention; future estimated and bias-corrected.",
    },
    "ndvi": {
        "standard_name": "normalized_difference_vegetation_index",
        "long_name": "NDVI",
        "units": "1",
        "historical_source_type": "hybrid_observed_proxy",
        "future_source_type": "proxy",
        "historical_folder_key": "ndvi",
        "future_folder_key": "ndvi",
        "method_note": "Observed NDVI where available; missing months filled with proxy predictions from climate inputs.",
    },
    "tws": {
        "standard_name": "terrestrial_water_storage_anomaly",
        "long_name": "total water storage proxy",
        "units": "1",
        "historical_source_type": "hybrid_observed_proxy",
        "future_source_type": "proxy",
        "historical_folder_key": "tws",
        "future_folder_key": "tws",
        "method_note": "Observed TWS where available; missing historical months and future months filled with proxy predictions from climate inputs.",
    },
}

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------
def extract_yyyymm(path: Path):
    stem = path.stem
    for token in stem.split("_"):
        if token.isdigit() and len(token) == 6:
            return token
    return None


def index_by_yyyymm(folder: Path, pattern: str = "*.tif"):
    out = {}
    if not folder.exists():
        return out
    for fp in sorted(folder.glob(pattern)):
        ym = extract_yyyymm(fp)
        if ym is not None:
            out[ym] = fp
    return out


def month_range(start: str, end: str):
    months = []
    y = int(start[:4])
    m = int(start[4:])

    while True:
        ym = f"{y:04d}{m:02d}"
        months.append(ym)

        if ym == end:
            break

        m += 1
        if m > 12:
            m = 1
            y += 1

    return months


def save_json(obj: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)


print("=" * 60)
print("MASTER INPUT DATASET SETUP")
print("=" * 60)
print(f"PROJECT ROOT   : {PROJECT_ROOT}")
print(f"RUN ID         : {RUN_ID}")
print(f"HIST PERIOD    : {HIST_START} -> {HIST_END}")
print(f"FUTURE PERIOD  : {FUTURE_START} -> {FUTURE_END}")
print(f"DATASET PATH   : {MASTER_DATASET_PATH}")
print(f"MANIFEST PATH  : {MASTER_MANIFEST_PATH}")
print(f"SCENARIOS      : {SCENARIOS}")
print(f"VARIABLES      : {list(VARIABLE_SPECS.keys())}")

MASTER INPUT DATASET SETUP
PROJECT ROOT   : C:\Projects\Infer RozviDrought
RUN ID         : master_inputs_dataset_20260323T090340Z
HIST PERIOD    : 198109 -> 202512
FUTURE PERIOD  : 202601 -> 205012
DATASET PATH   : C:\Projects\Infer RozviDrought\data\artifacts\datasets\master_inputs_dataset_20260323T090340Z.parquet
MANIFEST PATH  : C:\Projects\Infer RozviDrought\data\artifacts\manifests\master_inputs_dataset_20260323T090340Z_manifest.json
SCENARIOS      : ['ssp245', 'ssp370', 'ssp585']
VARIABLES      : ['t2m', 'd2m', 'sm', 'pet', 'ndvi', 'tws']


In [27]:
# ---------------------------------------------------------------------
# CELL 2: BUILD MASTER SOURCE REGISTRY
# One row per month x variable x scenario with exact source file
# Scans both .tif and .nc so false missing rows are avoided
# Assumes regenerated historical TWS now lives in:
#   PROXY_CORRECTED_DIR / "tws" / "historical"
# ---------------------------------------------------------------------

print("=" * 60)
print("BUILD MASTER SOURCE REGISTRY")
print("=" * 60)

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def build_month_file_index(folder: Path):
    out = {}

    if not folder.exists():
        return out

    for fp in sorted(folder.glob("*.tif")):
        ym = extract_yyyymm(fp)
        if ym is not None:
            out[ym] = fp

    for fp in sorted(folder.glob("*.nc")):
        ym = extract_yyyymm(fp)

        if ym is not None:
            out[ym] = fp
            continue

        try:
            import xarray as xr

            ds = xr.open_dataset(fp)

            if "time" in ds.coords:
                times = pd.to_datetime(ds["time"].values)

                for t in times:
                    ym_from_time = f"{t.year}{t.month:02d}"
                    if ym_from_time not in out:
                        out[ym_from_time] = fp

            ds.close()

        except Exception:
            continue

    return out


def source_type_for(var: str, period: str):
    spec = VARIABLE_SPECS[var]
    return (
        spec["historical_source_type"]
        if period == "historical"
        else spec["future_source_type"]
    )


# ------------------------------------------------------------
# BUILD MONTH LISTS
# ------------------------------------------------------------
hist_months = month_range(HIST_START, HIST_END)
future_months = month_range(FUTURE_START, FUTURE_END)

records = []

# ------------------------------------------------------------
# PRE-INDEX HISTORICAL SOURCES
# ------------------------------------------------------------
def merge_month_indexes(primary: Path, override: Path):
    out = {}

    # original tif months
    if primary.exists():
        for fp in sorted(primary.glob("*.tif")):
            ym = extract_yyyymm(fp)
            if ym is not None:
                out[ym] = fp

    # aligned tif months override / fill
    if override.exists():
        for fp in sorted(override.glob("*.tif")):
            ym = extract_yyyymm(fp)
            if ym is not None:
                out[ym] = fp

    return out

hist_indexes = {
    "t2m": merge_month_indexes(ERA5_DIR / "t2m", ALIGNED_DIR / "t2m"),
    "d2m": merge_month_indexes(ERA5_DIR / "d2m", ALIGNED_DIR / "d2m"),
    "sm": build_month_file_index(BASE_VARIABLES["sm"]),
    "pet": merge_month_indexes(ERA5_DIR / "pet", ALIGNED_DIR / "pet"),
    "ndvi": build_month_file_index(PROXY_HISTORICAL["ndvi"]),
    "tws": build_month_file_index(PROXY_HISTORICAL["tws"]),
}

# ------------------------------------------------------------
# PRE-INDEX FUTURE SOURCES
# ------------------------------------------------------------
future_indexes = {}

for scenario in SCENARIOS:
    future_indexes[scenario] = {
        "t2m": build_month_file_index(
            CMIP6_CORRECTED_DIR / scenario / VARIABLE_SPECS["t2m"]["future_folder_key"]
        ),
        "d2m": build_month_file_index(
            CMIP6_CORRECTED_DIR / scenario / VARIABLE_SPECS["d2m"]["future_folder_key"]
        ),
        "sm": build_month_file_index(
            CMIP6_CORRECTED_DIR / scenario / VARIABLE_SPECS["sm"]["future_folder_key"]
        ),
        "pet": build_month_file_index(
            CMIP6_CORRECTED_DIR / scenario / VARIABLE_SPECS["pet"]["future_folder_key"]
        ),
        "ndvi": build_month_file_index(
            PROXY_CORRECTED_DIR / VARIABLE_SPECS["ndvi"]["future_folder_key"] / scenario
        ),
        "tws": build_month_file_index(
            PROXY_CORRECTED_DIR / VARIABLE_SPECS["tws"]["future_folder_key"] / scenario
        ),
    }

# ------------------------------------------------------------
# HISTORICAL RECORDS
# ------------------------------------------------------------
for yyyymm in hist_months:
    for var in VARIABLE_SPECS.keys():

        fp = hist_indexes[var].get(yyyymm)

        records.append({
            "yyyymm": yyyymm,
            "period": "historical",
            "scenario": "historical",
            "variable": var,
            "standard_name": VARIABLE_SPECS[var]["standard_name"],
            "long_name": VARIABLE_SPECS[var]["long_name"],
            "units": VARIABLE_SPECS[var]["units"],
            "source_type": source_type_for(var, "historical"),
            "method_note": VARIABLE_SPECS[var]["method_note"],
            "source_file": str(fp) if fp is not None else None,
            "source_format": fp.suffix.lower() if fp is not None else None,
            "exists": fp is not None,
        })

# ------------------------------------------------------------
# FUTURE RECORDS
# ------------------------------------------------------------
for scenario in SCENARIOS:
    for yyyymm in future_months:
        for var in VARIABLE_SPECS.keys():

            fp = future_indexes[scenario][var].get(yyyymm)

            records.append({
                "yyyymm": yyyymm,
                "period": "future",
                "scenario": scenario,
                "variable": var,
                "standard_name": VARIABLE_SPECS[var]["standard_name"],
                "long_name": VARIABLE_SPECS[var]["long_name"],
                "units": VARIABLE_SPECS[var]["units"],
                "source_type": source_type_for(var, "future"),
                "method_note": VARIABLE_SPECS[var]["method_note"],
                "source_file": str(fp) if fp is not None else None,
                "source_format": fp.suffix.lower() if fp is not None else None,
                "exists": fp is not None,
            })

# ------------------------------------------------------------
# TO DATAFRAME
# ------------------------------------------------------------
source_registry = pd.DataFrame(records)

source_registry = source_registry.sort_values(
    ["period", "scenario", "yyyymm", "variable"]
).reset_index(drop=True)

missing_registry = source_registry.loc[
    ~source_registry["exists"]
].copy()

print(f"Registry rows        : {len(source_registry):,}")
print(f"Missing source rows  : {len(missing_registry):,}")

if len(missing_registry) > 0:
    print("\nFirst missing rows:")
    display(
        missing_registry[
            ["yyyymm", "period", "scenario", "variable", "source_type"]
        ].head(20)
    )
else:
    print("\nNo missing source files in registry.")

print("\nRegistry sample:")
display(source_registry.head(10))

BUILD MASTER SOURCE REGISTRY
Registry rows        : 8,592
Missing source rows  : 0

No missing source files in registry.

Registry sample:


,yyyymm,period,scenario,variable,standard_name,long_name,units,source_type,method_note,source_file,source_format,exists
0,202601,future,ssp245,d2m,dew_point_temperature,2 metre dew point temperature,K,bias_corrected_cmip6,Historical from ERA5-Land reference inputs; fu...,C:\Projects\Infer RozviDrought\rasters\correct...,.tif,True
1,202601,future,ssp245,ndvi,normalized_difference_vegetation_index,NDVI,1,proxy,Observed NDVI where available; missing months ...,C:\Projects\Infer RozviDrought\rasters\correct...,.tif,True
2,202601,future,ssp245,pet,potential_evapotranspiration,potential evapotranspiration,mm month-1,bias_corrected_cmip6,Historical from ERA5 reference inputs with pos...,C:\Projects\Infer RozviDrought\rasters\correct...,.tif,True
3,202601,future,ssp245,sm,soil_moisture,soil moisture,m3 m-3,bias_corrected_cmip6,Historical from reference inputs; future from ...,C:\Projects\Infer RozviDrought\rasters\correct...,.tif,True
4,202601,future,ssp245,t2m,air_temperature,2 metre air temperature,K,bias_corrected_cmip6,Historical from ERA5-Land reference inputs; fu...,C:\Projects\Infer RozviDrought\rasters\correct...,.tif,True
5,202601,future,ssp245,tws,terrestrial_water_storage_anomaly,total water storage proxy,1,proxy,Observed TWS where available; missing historic...,C:\Projects\Infer RozviDrought\rasters\correct...,.tif,True
6,202602,future,ssp245,d2m,dew_point_temperature,2 metre dew point temperature,K,bias_corrected_cmip6,Historical from ERA5-Land reference inputs; fu...,C:\Projects\Infer RozviDrought\rasters\correct...,.tif,True
7,202602,future,ssp245,ndvi,normalized_difference_vegetation_index,NDVI,1,proxy,Observed NDVI where available; missing months ...,C:\Projects\Infer RozviDrought\rasters\correct...,.tif,True
8,202602,future,ssp245,pet,potential_evapotranspiration,potential evapotranspiration,mm month-1,bias_corrected_cmip6,Historical from ERA5 reference inputs with pos...,C:\Projects\Infer RozviDrought\rasters\correct...,.tif,True
9,202602,future,ssp245,sm,soil_moisture,soil moisture,m3 m-3,bias_corrected_cmip6,Historical from reference inputs; future from ...,C:\Projects\Infer RozviDrought\rasters\correct...,.tif,True


In [29]:
# ---------------------------------------------------------------------
# CELL 2B: ALIGN FUTURE CMIP6 RASTERS TO TEMPLATE GRID
# - Aligns future tas/d2m/sm/pet to the 70x80 template
# - Saves to data/aligned_future/<scenario>/<var>/<var>_YYYYMM.tif
# - Updates source_registry in memory for Cell 3
# ---------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling

print("=" * 60)
print("ALIGN FUTURE CMIP6 RASTERS TO TEMPLATE GRID")
print("=" * 60)

# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------
if "source_registry" not in globals():
    raise RuntimeError("source_registry not found. Run Cell 2 first.")

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
ALIGNED_FUTURE_DIR = DATA_DIR / "aligned_future"
ALIGNED_FUTURE_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# TEMPLATE GRID
# ------------------------------------------------------------
template_path = BASE_VARIABLES["sm"] / f"sm_{HIST_START}.tif"

with rasterio.open(template_path) as tmpl:
    template_meta = tmpl.meta.copy()
    template_crs = tmpl.crs
    template_transform = tmpl.transform
    template_height = tmpl.height
    template_width = tmpl.width

template_meta.update(
    dtype="float32",
    count=1,
    compress="lzw",
    nodata=np.nan,
)

print(f"Template path : {template_path}")
print(f"Template grid : ({template_height}, {template_width})")
print(f"Template CRS  : {template_crs}")

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def align_tif_to_template(src_path: Path, out_path: Path):
    with rasterio.open(src_path) as src:
        arr = src.read(1).astype(np.float32)
        nodata = src.nodata

        if nodata is not None:
            arr = np.where(arr == nodata, np.nan, arr)

        dst = np.full((template_height, template_width), np.nan, dtype=np.float32)

        reproject(
            source=arr,
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=template_transform,
            dst_crs=template_crs,
            src_nodata=np.nan,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear,
        )

    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(out_path, "w", **template_meta) as dst_file:
        dst_file.write(dst, 1)


# future source folders
future_source_map = {
    "t2m": "tas",
    "d2m": "d2m",
    "sm": "sm",
    "pet": "pet",
}

summary = {
    "processed": 0,
    "skipped_existing": 0,
    "errors": [],
}

# ------------------------------------------------------------
# STEP A: ALIGN FUTURE FILES
# ------------------------------------------------------------
for scenario in SCENARIOS:
    for var, src_subfolder in future_source_map.items():

        src_folder = CMIP6_CORRECTED_DIR / scenario / src_subfolder
        out_folder = ALIGNED_FUTURE_DIR / scenario / var
        out_folder.mkdir(parents=True, exist_ok=True)

        for src_path in sorted(src_folder.glob("*.tif")):
            ym = extract_yyyymm(src_path)
            if ym is None:
                continue

            out_path = out_folder / f"{var}_{ym}.tif"

            if out_path.exists():
                summary["skipped_existing"] += 1
                continue

            try:
                align_tif_to_template(src_path, out_path)
                summary["processed"] += 1
            except Exception as e:
                summary["errors"].append({
                    "scenario": scenario,
                    "variable": var,
                    "source_file": str(src_path),
                    "reason": f"{type(e).__name__}: {str(e)[:250]}",
                })

print(f"Aligned new files : {summary['processed']}")
print(f"Skipped existing  : {summary['skipped_existing']}")
print(f"Alignment errors  : {len(summary['errors'])}")

if summary["errors"]:
    print("\nFirst alignment errors:")
    display(pd.DataFrame(summary["errors"]).head(10))

# ------------------------------------------------------------
# STEP B: UPDATE source_registry IN MEMORY
# ------------------------------------------------------------
updated = 0

for scenario in SCENARIOS:
    for var in future_source_map.keys():
        out_folder = ALIGNED_FUTURE_DIR / scenario / var

        for fp in sorted(out_folder.glob("*.tif")):
            ym = extract_yyyymm(fp)
            if ym is None:
                continue

            mask = (
                (source_registry["period"] == "future") &
                (source_registry["scenario"] == scenario) &
                (source_registry["variable"] == var) &
                (source_registry["yyyymm"] == ym)
            )

            if mask.any():
                source_registry.loc[mask, "source_file"] = str(fp)
                source_registry.loc[mask, "source_format"] = ".tif"
                source_registry.loc[mask, "exists"] = True
                updated += int(mask.sum())

print(f"\nUpdated source_registry rows : {updated:,}")

# ------------------------------------------------------------
# STEP C: QUICK VALIDATION
# ------------------------------------------------------------
future_check = source_registry.loc[
    (source_registry["period"] == "future") &
    (source_registry["variable"].isin(["t2m", "d2m", "sm", "pet"]))
].copy()

print(f"Future aligned rows checked : {len(future_check):,}")
print(f"Missing after update        : {(~future_check['exists']).sum():,}")

sample_fp = Path(
    future_check.loc[
        (future_check["variable"] == "t2m") &
        (future_check["scenario"] == SCENARIOS[0]),
        "source_file"
    ].iloc[0]
)

with rasterio.open(sample_fp) as src:
    print(f"Sample aligned future file  : {sample_fp}")
    print(f"Sample shape                : ({src.height}, {src.width})")
    print(f"Sample CRS                  : {src.crs}")

ALIGN FUTURE CMIP6 RASTERS TO TEMPLATE GRID
Template path : C:\Projects\Infer RozviDrought\data\simulated\era5_land\sm\sm_198109.tif
Template grid : (70, 80)
Template CRS  : EPSG:4326
Aligned new files : 3600
Skipped existing  : 0
Alignment errors  : 0

Updated source_registry rows : 3,600
Future aligned rows checked : 3,600
Missing after update        : 0
Sample aligned future file  : C:\Projects\Infer RozviDrought\data\aligned_future\ssp245\t2m\t2m_202601.tif
Sample shape                : (70, 80)
Sample CRS                  : EPSG:4326


In [30]:
# ---------------------------------------------------------------------
# CELL 3: BUILD MASTER INPUT DATASET + MANIFEST + QA
# ---------------------------------------------------------------------

import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import rasterio

print("=" * 60)
print("BUILD MASTER INPUT DATASET + MANIFEST + QA")
print("=" * 60)

# ------------------------------------------------------------
# OUTPUT PATHS
# ------------------------------------------------------------

OUTPUT_DIR = DATA_DIR / "master_inputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

DATASET_PATH = OUTPUT_DIR / f"master_inputs_{timestamp}.parquet"
MANIFEST_PATH = OUTPUT_DIR / f"master_inputs_manifest_{timestamp}.json"

# ------------------------------------------------------------
# TEMPLATE GRID
# ------------------------------------------------------------

template_row = source_registry.iloc[0]
template_path = Path(template_row["source_file"])

with rasterio.open(template_path) as src:
    template_height = src.height
    template_width = src.width
    template_crs = src.crs
    template_transform = src.transform

pixel_count = template_height * template_width

print(f"Template path : {template_path}")
print(f"Grid shape    : ({template_height}, {template_width})")
print(f"Pixel count   : {pixel_count:,}")

# ------------------------------------------------------------
# RASTER READER
# ------------------------------------------------------------

def read_raster_flat(path: Path):

    with rasterio.open(path) as src:

        arr = src.read(1).astype(np.float32)

        nodata = src.nodata

        if nodata is not None:
            arr[arr == nodata] = np.nan

        return (
            arr.flatten(),
            src.height,
            src.width,
            src.crs,
            src.transform,
        )

# ------------------------------------------------------------
# GROUP REGISTRY
# ------------------------------------------------------------

value_vars = [
    "t2m",
    "d2m",
    "sm",
    "pet",
    "ndvi",
    "tws",
]

groups = (
    source_registry
    .sort_values(["scenario", "yyyymm", "variable"])
    .groupby(["scenario", "yyyymm"])
)

dataset_parts = []

total_groups = len(groups)

print(f"Total groups  : {total_groups:,}")

# ------------------------------------------------------------
# BUILD DATASET
# ------------------------------------------------------------

for i, ((scenario, yyyymm), grp) in enumerate(groups):

    row_df = pd.DataFrame()

    row_df["scenario"] = [scenario] * pixel_count
    row_df["yyyymm"] = [yyyymm] * pixel_count

    for var in value_vars:

        fp = grp.loc[
            grp["variable"] == var,
            "source_file"
        ].iloc[0]

        flat, h, w, crs, transform = read_raster_flat(Path(fp))

        if h != template_height or w != template_width:

            raise RuntimeError(
                f"{var} {yyyymm} {scenario}: "
                f"grid mismatch. "
                f"Expected ({template_height}, {template_width}), "
                f"got ({h}, {w})"
            )

        row_df[var] = flat

    dataset_parts.append(row_df)

    if (i + 1) % 50 == 0:

        print(
            f"Built groups: {i + 1:,} / {total_groups:,}"
        )

# ------------------------------------------------------------
# CONCATENATE
# ------------------------------------------------------------

master_df = pd.concat(
    dataset_parts,
    ignore_index=True,
)

print("\nFinal dataset shape:")
print(master_df.shape)

# ------------------------------------------------------------
# SAVE DATASET
# ------------------------------------------------------------

master_df.to_parquet(
    DATASET_PATH,
    index=False,
)

print("\nSaved dataset:")
print(DATASET_PATH)

# ------------------------------------------------------------
# BUILD MANIFEST
# ------------------------------------------------------------

manifest = {

    "dataset_path": str(DATASET_PATH),

    "created_utc": timestamp,

    "grid": {
        "height": template_height,
        "width": template_width,
        "pixel_count": pixel_count,
        "crs": str(template_crs),
    },

    "time_range": {
        "start": str(master_df["yyyymm"].min()),
        "end": str(master_df["yyyymm"].max()),
    },

    "scenarios": sorted(
        master_df["scenario"].unique().tolist()
    ),

    "variables": value_vars,

    "row_count": int(len(master_df)),

    "column_count": int(len(master_df.columns)),

}

with open(MANIFEST_PATH, "w") as f:

    json.dump(
        manifest,
        f,
        indent=2,
    )

print("\nSaved manifest:")
print(MANIFEST_PATH)

# ------------------------------------------------------------
# QA CHECKS
# ------------------------------------------------------------

print("\nQA CHECKS")

print("Missing values per variable:")

print(
    master_df[value_vars]
    .isna()
    .sum()
)

print("\nDataset ready for downstream use.")

BUILD MASTER INPUT DATASET + MANIFEST + QA
Template path : C:\Projects\Infer RozviDrought\data\aligned_future\ssp245\d2m\d2m_202601.tif
Grid shape    : (70, 80)
Pixel count   : 5,600
Total groups  : 1,432


C:\Users\USER\AppData\Local\Temp\ipykernel_14864\4244845637.py:24: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")


Built groups: 50 / 1,432
Built groups: 100 / 1,432
Built groups: 150 / 1,432
Built groups: 200 / 1,432
Built groups: 250 / 1,432
Built groups: 300 / 1,432
Built groups: 350 / 1,432
Built groups: 400 / 1,432
Built groups: 450 / 1,432
Built groups: 500 / 1,432
Built groups: 550 / 1,432
Built groups: 600 / 1,432
Built groups: 650 / 1,432
Built groups: 700 / 1,432
Built groups: 750 / 1,432
Built groups: 800 / 1,432
Built groups: 850 / 1,432
Built groups: 900 / 1,432
Built groups: 950 / 1,432
Built groups: 1,000 / 1,432
Built groups: 1,050 / 1,432
Built groups: 1,100 / 1,432
Built groups: 1,150 / 1,432
Built groups: 1,200 / 1,432
Built groups: 1,250 / 1,432
Built groups: 1,300 / 1,432
Built groups: 1,350 / 1,432
Built groups: 1,400 / 1,432

Final dataset shape:
(8019200, 8)

Saved dataset:
C:\Projects\Infer RozviDrought\data\master_inputs\master_inputs_20260323T112832Z.parquet

Saved manifest:
C:\Projects\Infer RozviDrought\data\master_inputs\master_inputs_manifest_20260323T112832Z.json

QA

In [31]:
master_df.groupby("scenario")["yyyymm"].nunique()

scenario
historical    532
ssp245        300
ssp370        300
ssp585        300
Name: yyyymm, dtype: int64